# DeepSeek OCR in Google Colab

<a href="https://colab.research.google.com/github/mark-baumann/ocr-erkennung/blob/claude/deepseek-ocr-streamlit-46tyl3/ocr_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Handschrift- und Dokument-Erkennung (OCR) ohne App

Dieses Notebook erkennt Texte — auch Handschrift — in gescannten Dokumenten, Bildern und PDFs mit **DeepSeek-OCR**, direkt in Google Colab. Keine lokale Installation, keine Streamlit-App, keine GUI.

**So geht's:**

1. Laufzeit > Laufzeittyp ändern → **GPU (T4)** wählen.
2. Alle Zellen der Reihe nach ausführen (Zelle 1 installiert das Projekt und die GPU-Abhängigkeiten).
3. In Zelle 2 ein Bild (PNG/JPG) oder ein PDF hochladen.
4. Das Ergebnis (Markdown, Text, Layout-JSON) wird als ZIP heruntergeladen.

# DeepSeek OCR in Google Colab

Aktiviere in Colab unter Laufzeit > Laufzeittyp aendern eine NVIDIA-GPU und fuehre die Zellen der Reihe nach aus.

In [ ]:
import subprocess
import sys
from pathlib import Path
REPO_ROOT = Path('/content/ocr-erkennung')
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', 'claude/deepseek-ocr-streamlit-46tyl3', '--depth', '1', 'https://github.com/mark-baumann/ocr-erkennung.git', str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements-gpu.txt')], check=True)
sys.path.insert(0, str(REPO_ROOT))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Keine NVIDIA-GPU gefunden. Aktiviere eine GPU-Laufzeit in Colab.')
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Keine Bild- oder PDF-Datei hochgeladen.')
INPUT_DIR = Path('/content/ocr-input')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
input_paths = []
for name, data in uploaded.items():
    path = INPUT_DIR / Path(name).name
    path.write_bytes(data)
    input_paths.append(path)
print('Hochgeladen:', ', '.join(path.name for path in input_paths))

In [ ]:
import json
import shutil
from dsocr.config import PROMPTS, Settings, get_mode
from dsocr.engines import create_engine
from dsocr.pipeline import resolve_prompt, run
from dsocr.preprocess.loader import load_document
OUTPUT_DIR = Path('/content/dsocr-out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
settings = Settings(model_path='deepseek-ai/DeepSeek-OCR', backend='transformers')
pages = []
for path in input_paths:
    document = load_document(path.name, path.read_bytes(), dpi=settings.pdf_dpi, max_pages=settings.max_pages, start_index=len(pages))
    pages.extend(document.pages)
engine = create_engine('transformers', settings)
result = run(pages, engine, resolve_prompt(PROMPTS['markdown']), get_mode('gundam'), settings=settings, batch=False)
(OUTPUT_DIR / 'document.md').write_text(result.combined_markdown(), encoding='utf-8')
(OUTPUT_DIR / 'document.txt').write_text(result.combined_text(), encoding='utf-8')
(OUTPUT_DIR / 'layout.json').write_text(json.dumps(result.to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
archive = shutil.make_archive('/content/dsocr-result', 'zip', OUTPUT_DIR)
print(f'Seiten: {result.summary.pages}; Fehler: {result.summary.failed}')
print(f'Ergebnis: {archive}')

In [ ]:
files.download('/content/dsocr-result.zip')